In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


In [ ]:
!pip install -q  peft evaluate


In [ ]:
!pip show transformers datasets peft accelerate evaluate | grep Version


A lightweight, fast, and flexible library to load and work with large datasets.

Provides standardized access to thousands of ready-to-use datasets (text, images, audio, tabular).

Optimized for streaming large files without requiring them to fully fit in memory.

Integrates seamlessly with PyTorch, TensorFlow, and JAX.
The datasets library by Hugging Face makes it easy to load, preprocess, and share datasets for machine learning. It’s fast and memory-efficient (built on Apache Arrow), supports streaming huge datasets, and comes with thousands of ready-to-use datasets from the Hugging Face Hub. You can easily map, filter, shuffle, and batch data, and it integrates seamlessly with PyTorch, TensorFlow, and JAX.

In [ ]:
# Load dataset into Hugging Face

from datasets import load_dataset


dataset = load_dataset("/kaggle/input/nlp-getting-started/", data_files={
    "train": "train.csv",
    "test": "test.csv"
})

# Optional: split train into train/validation
dataset = dataset["train"].train_test_split(test_size=0.05, seed=46)

print(dataset)

In [ ]:
dataset["test"]

In [ ]:
dataset["train"]

In [ ]:
dataset["test"][0]

In [ ]:
dataset["train"][0]

### Flan-T5 Overview
- **Flan-T5** is an instruction-tuned version of Google's [T5 (Text-to-Text Transfer Transformer)](https://huggingface.co/docs/transformers/model_doc/t5).  
- It improves **zero-shot and few-shot generalization** by training on a wide range of tasks phrased as natural language instructions.  
- Available on Hugging Face in multiple sizes:  
  - [Flan-T5 Small (80M)](https://huggingface.co/google/flan-t5-small)  
  - [Flan-T5 Base (250M)](https://huggingface.co/google/flan-t5-base)  
  - [Flan-T5 Large (780M)](https://huggingface.co/google/flan-t5-large)  
  - [Flan-T5 XL (3B)](https://huggingface.co/google/flan-t5-xl)  
  - [Flan-T5 XXL (11B)](https://huggingface.co/google/flan-t5-xxl)  

### Vanilla T5 vs Flan-T5
- **Vanilla T5**: Original model trained on the *C4 dataset* (a cleaned version of Common Crawl).  
  - Treated all NLP tasks as *text-to-text* problems (e.g., "Translate English to German: Hello" → "Hallo").  
  - Very powerful but required task-specific prompt formatting.  

- **Flan-T5**: Builds on T5 by **fine-tuning on many instruction-formatted datasets**,  
  making it much more user-friendly and effective for real-world prompts without heavy reformatting.  


In [ ]:
from transformers import AutoTokenizer
#model_name = "google/flan-t5-small"
#model_name = "google/flan-t5-base"
model_name = "google/flan-t5-large"
#model_name = "google/flan-t5-xl"
#model_name = "google/flan-t5-xxl"
tokenizer = AutoTokenizer.from_pretrained(model_name)


# Transformers

transformers is a Python library by Hugging Face.

It provides pre-trained state-of-the-art NLP and vision models (like BERT, GPT-2, T5, Flan-T5, etc.).

You can load models and tokenizers with just one line of code.

# AutoTokenizer

A class in transformers that automatically selects the correct tokenizer for the model you’re loading.

Tokenizers convert text → tokens (numbers) and back again.

Different models (BERT, GPT, T5, etc.) use slightly different tokenization rules, but AutoTokenizer figures that out for you.

Explore different tokenizers - https://huggingface.co/spaces/Xenova/the-tokenizer-playground

In [ ]:
# Example text
text = "Flan-T5 makes instruction following easier."

# 1. Tokenize text → token IDs
tokens = tokenizer(text)
print("Token IDs:", tokens["input_ids"])

# 2. Convert token IDs back to text
decoded_text = tokenizer.decode(tokens["input_ids"])
print("Decoded text:", decoded_text)

# 3. View token-by-token breakdown
tokens_decoded = [tokenizer.decode([tid]) for tid in tokens["input_ids"]]
print("Tokens:", tokens_decoded)

# 4. Batch tokenization
batch = tokenizer(["Hello world!", "Flan-T5 is great!"], padding=True, truncation=True, return_tensors="pt")
print("Batch input IDs:\n", batch["input_ids"])

# 5. Special tokens
print("Pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("EOS token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("Vocabulary size:", tokenizer.vocab_size)


In [ ]:
def make_prompt(text, keyward, location):
    return f"""Classify whether the following tweet is about a disaster or not.
Tweet: "{text}"
Keyword: "{keyward}"
Location: "{location}"
Answer with "yes" or "no" only."""


In [ ]:
max_input_length = 128
max_target_length = 3

def preprocess_function(examples):
    # Build prompts and simple yes/no targets
    inputs = [
        make_prompt(text, keyword, location)
        for text, keyword, location in zip(examples["text"], examples["keyword"], examples["location"])
    ]
    
    targets = ["yes" if l == 1 else "no" for l in examples["target"]]

    # Tokenize inputs and targets
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length",
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=max_target_length,
            truncation=True,
            padding="max_length",
        )["input_ids"]

    # Replace padding token ids with -100 (ignore index)
    labels = [
        [token if token != tokenizer.pad_token_id else -100 for token in label_seq]
        for label_seq in labels
    ]
    model_inputs["labels"] = labels
    return model_inputs


tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)
tokenized_datasets.set_format("torch")


In [ ]:
tokenized_datasets['train']

In [ ]:
tokenized_datasets['train'][0]['input_ids']

In [ ]:
tokenizer.decode(tokenized_datasets['train'][68]['input_ids'])

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Input Text → Tokenizer → Encoder (stack of Transformer blocks) 
           → Decoder (stack of Transformer blocks with cross-attention to encoder) 
           → Linear + Softmax → Output tokens


In [ ]:
model

TODO - explore the actual model in detail 

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1
)
model = get_peft_model(model, lora_config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.print_trainable_parameters()



In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import evaluate

accuracy = evaluate.load("accuracy")

import numpy as np
# Assuming 'accuracy' is loaded, e.g., from evaluate import load; accuracy = load("accuracy")

import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # If predictions is a tuple (logits, ...), take the first part
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Argmax over vocabulary dimension
    pred_ids = np.argmax(predictions, axis=-1)

    # Replace -100 in labels so we can decode properly
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Decode both predictions and labels
    decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Normalize text
    decoded_preds = [p.strip().lower() for p in decoded_preds]
    decoded_labels = [l.strip().lower() for l in decoded_labels]

    # Map yes/no → 1/0
    binary_preds = [1 if p == "yes" else 0 for p in decoded_preds]
    binary_labels = [1 if l == "yes" else 0 for l in decoded_labels]

    # Compute accuracy
    return accuracy.compute(predictions=binary_preds, references=binary_labels)


args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-disaster",
    eval_strategy="steps",
    eval_steps=100,
    logging_steps=50,
    save_total_limit=1,
    learning_rate=5e-5,
    per_device_train_batch_size=16, # Reduced batch size
    per_device_eval_batch_size=16,  # Reduced batch size
    num_train_epochs=1,
    fp16=False,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    report_to=[],  # <-- disable W&B, TensorBoard, etc.

)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    
)

In [ ]:
print(tokenized_datasets["test"][0])


In [ ]:
trainer.args.dataloader_num_workers = 0 

In [ ]:
result = trainer.predict(tokenized_datasets["test"])

In [ ]:
result

In [ ]:
# Evaluate the model before training
print("Evaluation before training:")
eval_results_before = trainer.evaluate(tokenized_datasets["test"])
print(eval_results_before)

In [ ]:
trainer.train()

In [ ]:
trainer.train()

In [ ]:
trainer.train()

In [ ]:
# After training (assuming you have already run trainer.train()), you can evaluate again
print("\nEvaluation after training:")
eval_results_after = trainer.evaluate(tokenized_datasets["test"])
print(eval_results_after)

In [ ]:
test_text = "Breaking news: major earthquake hits downtown"
#[f"Classify if this tweet is about a disaster, output yes or no only: {t}" for t in examples["text"]]
input_ids = tokenizer(make_prompt(test_text, "", ""),
                      return_tensors="pt").input_ids.to(device)
outputs = model.generate(input_ids=input_ids, max_length=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
outputs

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")
# Make sure model is ready
model.eval()
model.to("cuda")

# ==== 1. Preview first few samples from the training/test data ====
print("\n=== Reviewing sample predictions ===\n")
for idx, row in test_df.head(50).iterrows():
    input_text = make_prompt(row['text'], row["keyword"], row["location"])
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_length=2)

    prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(f"Tweet: {row['text']}")
    print(f"Prediction: {prediction}")
    print("-" * 80)

# ==== 2. Full inference for submission.csv ====



predictions = []
batch_size = 32

print("\n=== Generating predictions for submission.csv ===\n")
from tqdm import tqdm
import torch

predictions = []
batch_size = 16  # adjust as needed

print("\n=== Generating predictions for submission.csv ===\n")
for i in tqdm(range(0, len(test_df), batch_size)):
    batch_texts = [
        make_prompt(t, k, l)
        for t, k, l in zip(
            test_df["text"][i:i+batch_size],
            test_df["keyword"][i:i+batch_size],
            test_df["location"][i:i+batch_size]
        )
    ]

    # Tokenize and move inputs to GPU safely
    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    inputs = {k: v.cuda() for k, v in inputs.items()}  # move each tensor to GPU

    # Generate predictions
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_length=2)

    # Decode predictions
    decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

    # Convert "yes"/"no" to 1/0
    preds = [1 if "yes" in p.lower() else 0 for p in decoded]
    predictions.extend(preds)

# Create submission file
submission = pd.DataFrame({
    "id": test_df["id"],
    "target": predictions
})
submission.to_csv("submission.csv", index=False)

print("\n✅ submission.csv generated with", len(submission), "rows")


In [ ]:
submission['target'].describe()